<a href="https://colab.research.google.com/github/Kard00/Machine_Learning/blob/main/primo_do_chat_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, Flatten
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences # Import pad_sequences
from tensorflow.keras.utils import to_categorical

# 1. DADOS DE TREINO - é aqui que ela "aprende" o que existe no mundo
frases = [
    "eu gosto de pizza",
    "eu gosto de pizza",
    "eu gosto de pizza",
    "eu gosto de star wars",
    "eu gosto de star wars",
    "eu gosto de machine learning",
    "eu gosto de cachorro",
    "eu gosto de café"
]
# Repeti "pizza" e "star wars" de propósito: a rede vai achar que é mais provável

# 2. TRANSFORMA TEXTO EM NÚMERO - rede neural não lê letra, só número
tokenizer = Tokenizer()
tokenizer.fit_on_texts(frases) # Cria um dicionário: eu=1, gosto=2, de=3, pizza=4...
print("Dicionário criado:", tokenizer.word_index)

# Converte as frases em sequências de números
sequencias = tokenizer.texts_to_sequences(frases)
# "eu gosto de pizza" vira [1, 2, 3, 4]

# 3. SEPARA PERGUNTA E RESPOSTA
# Pergunta X: "eu gosto de" = [1, 2, 3]
# Resposta y: "pizza" = [4]
X, y = [], []
for seq in sequencias:
    X.append(seq[:-1]) # Pega tudo menos a última palavra
    y.append(seq[-1]) # Pega só a última palavra

# CORREÇÃO: Pad sequences to uniform length before converting to numpy array
max_len = 3 # All 'questions' are "eu gosto de" (3 words)
X = pad_sequences(X, maxlen=max_len, padding='pre') # Pad sequences to the left

X = np.array(X) # Now this will work as all sequences in X have the same length
y = np.array(y)

# 4. CONVERTE A RESPOSTA PRA FORMATO QUE A REDE ENTENDE
vocab_size = len(tokenizer.word_index) + 1
y = to_categorical(y, num_classes=vocab_size)

# 5. CRIA O "CÉREBRO" - 3 camadas bem burras
modelo = Sequential()
modelo.add(Embedding(input_dim=vocab_size, output_dim=10, input_length=max_len)) # Traduz número pra vetor, input_length adjusted
modelo.add(Flatten()) # Achata tudo
modelo.add(Dense(vocab_size, activation='softmax')) # Camada que escolhe a próxima palavra

modelo.compile(loss='categorical_crossentropy', optimizer='adam')

# 6. TREINA - aqui ela ajusta os "botões" internos 500 vezes
print("\nTreinando... a rede está errando e se corrigindo igual bebê aprendendo a falar")
modelo.fit(X, y, epochs=500, verbose=0)
print("Treino finalizado!")

# 7. TESTA - pergunta pra rede: "eu gosto de ___"
teste = tokenizer.texts_to_sequences(["eu gosto de"])
teste = pad_sequences(teste, maxlen=max_len, padding='pre') # Pad the test sequence too
teste = np.array(teste)

previsao = modelo.predict(teste, verbose=0)
palavra_id = np.argmax(previsao) # Pega o índice com maior probabilidade

# Converte número de volta pra palavra
for palavra, index in tokenizer.word_index.items():
    if index == palavra_id:
        print(f"\nResultado: 'eu gosto de {palavra}'")
        break

# 8. VÊ AS PROBABILIDADES - é assim que ela "pensa"
print("\nProbabilidade de cada palavra ser a próxima:")
for palavra, index in tokenizer.word_index.items():
    if index < len(previsao[0]):
        print(f"{palavra}: {previsao[0][index]*100:.1f}%")

Dicionário criado: {'eu': 1, 'gosto': 2, 'de': 3, 'pizza': 4, 'star': 5, 'wars': 6, 'machine': 7, 'learning': 8, 'cachorro': 9, 'café': 10}

Treinando... a rede está errando e se corrigindo igual bebê aprendendo a falar


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Treino finalizado!

Resultado: 'eu gosto de pizza'

Probabilidade de cada palavra ser a próxima:
eu: 0.1%
gosto: 0.1%
de: 0.1%
pizza: 59.3%
star: 0.1%
wars: 0.3%
machine: 0.1%
learning: 0.1%
cachorro: 19.9%
café: 19.9%


In [3]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, Flatten
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

# 1. DADOS DE TREINO
frases = [
    "eu gosto de pizza",
    "eu gosto de pizza",
    "eu gosto de pizza",
    "eu gosto de star wars",
    "eu gosto de star wars",
    "eu gosto de machine learning",
    "eu gosto de cachorro",
    "eu gosto de café"
]

# 2. TRANSFORMA TEXTO EM NÚMERO
tokenizer = Tokenizer()
tokenizer.fit_on_texts(frases)
print("Dicionário criado:", tokenizer.word_index)

sequencias = tokenizer.texts_to_sequences(frases)

# 3. SEPARA PERGUNTA E RESPOSTA
X, y = [], []
for seq in sequencias:
    X.append(seq[:-1]) # "eu gosto de"
    y.append(seq[-1]) # "pizza"

# 4. AQUI ESTAVA O ERRO - AGORA CORRIGIDO
max_len = 3 # sabemos que "eu gosto de" tem 3 palavras
X = pad_sequences(X, maxlen=max_len, padding='pre') # Garante que todo mundo tenha tamanho 3
X = np.array(X, dtype='int32') # Força ser número inteiro
y = np.array(y, dtype='int32')

# 5. CONVERTE A RESPOSTA PRA FORMATO QUE A REDE ENTENDE
vocab_size = len(tokenizer.word_index) + 1
y = to_categorical(y, num_classes=vocab_size)

# 6. CRIA O "CÉREBRO"
modelo = Sequential()
modelo.add(Embedding(input_dim=vocab_size, output_dim=10, input_length=max_len))
modelo.add(Flatten())
modelo.add(Dense(vocab_size, activation='softmax'))

modelo.compile(loss='categorical_crossentropy', optimizer='adam')

# 7. TREINA
print("\nTreinando... a rede está errando e se corrigindo igual bebê aprendendo a falar")
modelo.fit(X, y, epochs=500, verbose=0)
print("Treino finalizado!")

# 8. TESTA
teste = tokenizer.texts_to_sequences(["eu gosto de"])
teste = pad_sequences(teste, maxlen=max_len, padding='pre')
teste = np.array(teste, dtype='int32')

previsao = modelo.predict(teste, verbose=0)
palavra_id = np.argmax(previsao)

for palavra, index in tokenizer.word_index.items():
    if index == palavra_id:
        print(f"\nResultado: 'eu gosto de {palavra}'")
        break

# 9. VÊ AS PROBABILIDADES
print("\nProbabilidade de cada palavra ser a próxima:")
for palavra, index in tokenizer.word_index.items():
    if index < len(previsao[0]):
        print(f"{palavra}: {previsao[0][index]*100:.1f}%")

Dicionário criado: {'eu': 1, 'gosto': 2, 'de': 3, 'pizza': 4, 'star': 5, 'wars': 6, 'machine': 7, 'learning': 8, 'cachorro': 9, 'café': 10}

Treinando... a rede está errando e se corrigindo igual bebê aprendendo a falar
Treino finalizado!

Resultado: 'eu gosto de pizza'

Probabilidade de cada palavra ser a próxima:
eu: 0.1%
gosto: 0.1%
de: 0.1%
pizza: 59.1%
star: 0.1%
wars: 0.5%
machine: 0.2%
learning: 0.1%
cachorro: 19.8%
café: 19.9%


In [ ]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, Flatten
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

# 1. DADOS DE TREINO
frases = [
    "darth vader usa sabre",
    "darth vader usa capacete",
    "darth vader usa capa",
    "darth vader usa a força",
    "luke usa sabre"
]

# 2. TRANSFORMA TEXTO EM NÚMERO
tokenizer = Tokenizer()
tokenizer.fit_on_texts(frases)
print("Dicionário criado:", tokenizer.word_index)

sequencias = tokenizer.texts_to_sequences(frases)

# 3. SEPARA PERGUNTA E RESPOSTA
X, y = [], []
for seq in sequencias:
    X.append(seq[:-1]) # "eu gosto de"
    y.append(seq[-1]) # "pizza"

# 4. AQUI ESTAVA O ERRO - AGORA CORRIGIDO
max_len = 3 # sabemos que "eu gosto de" tem 3 palavras
X = pad_sequences(X, maxlen=max_len, padding='pre') # Garante que todo mundo tenha tamanho 3
X = np.array(X, dtype='int32') # Força ser número inteiro
y = np.array(y, dtype='int32')

# 5. CONVERTE A RESPOSTA PRA FORMATO QUE A REDE ENTENDE
vocab_size = len(tokenizer.word_index) + 1
y = to_categorical(y, num_classes=vocab_size)

# 6. CRIA O "CÉREBRO"
modelo = Sequential()
modelo.add(Embedding(input_dim=vocab_size, output_dim=10, input_length=max_len))
modelo.add(Flatten())
modelo.add(Dense(vocab_size, activation='softmax'))

modelo.compile(loss='categorical_crossentropy', optimizer='adam')

# 7. TREINA
print("\nTreinando... a rede está errando e se corrigindo igual bebê aprendendo a falar")
modelo.fit(X, y, epochs=500, verbose=0)
print("Treino finalizado!")

# 8. TESTA
teste = tokenizer.texts_to_sequences(["darth vader usa"])
teste = pad_sequences(teste, maxlen=max_len, padding='pre')
teste = np.array(teste, dtype='int32')

previsao = modelo.predict(teste, verbose=0)
palavra_id = np.argmax(previsao)

for palavra, index in tokenizer.word_index.items():
    if index == palavra_id:
        print(f"\nResultado: 'darth vader usa {palavra}'")
        break

# 9. VÊ AS PROBABILIDADES
print("\nProbabilidade de cada palavra ser a próxima:")
for palavra, index in tokenizer.word_index.items():
    if index < len(previsao[0]):
        print(f"{palavra}: {previsao[0][index]*100:.1f}%")

Dicionário criado: {'usa': 1, 'darth': 2, 'vader': 3, 'sabre': 4, 'capacete': 5, 'capa': 6, 'a': 7, 'força': 8, 'luke': 9}

Treinando... a rede está errando e se corrigindo igual bebê aprendendo a falar
Treino finalizado!

Resultado: 'darth vader usa sabre'

Probabilidade de cada palavra ser a próxima:
usa: 0.1%
darth: 0.2%
vader: 0.2%
sabre: 33.6%
capacete: 32.5%
capa: 32.7%
a: 0.1%
força: 0.3%
luke: 0.1%
